In [1]:
class GinzburgCohomologyCalculator:
    """
    A class for calculation of the cohomology of a Ginzburg dga
    """
    def __init__(self, a, b, c, custom_W=None, potential_type='generic', seed=None):
        self.a = a
        self.b = b
        self.c = c
        self.elements = {}
        self.x_ids, self.y_ids, self.z_ids = [], [], []
        self.x_s_ids, self.y_s_ids, self.z_s_ids = [], [], []
        self.t_ids = []
        self.basis_cache = {}
        
        # Quiverの構造構築
        self._build_quiver()
        
        # ポテンシャル W の設定
        # custom_W が渡された場合はそれを採用し、無ければ自動生成する
        if custom_W is not None:
            self.W_coeffs = custom_W
            self.potential_type = 'custom'
        else:
            self.potential_type = potential_type
            self.W_coeffs = self.generate_potential(potential_type, seed)

    def _build_quiver(self):
        
        el_idx = 0
        a, b, c = self.a, self.b, self.c
        
        for _ in range(a): self.elements[el_idx] = {'s': 1, 'e': 2, 'deg': 0, 'len': 1}; self.x_ids.append(el_idx); el_idx += 1
        for _ in range(b): self.elements[el_idx] = {'s': 2, 'e': 3, 'deg': 0, 'len': 1}; self.y_ids.append(el_idx); el_idx += 1
        for _ in range(c): self.elements[el_idx] = {'s': 3, 'e': 1, 'deg': 0, 'len': 1}; self.z_ids.append(el_idx); el_idx += 1
            
        for _ in range(a): self.elements[el_idx] = {'s': 2, 'e': 1, 'deg': -1, 'len': 2}; self.x_s_ids.append(el_idx); el_idx += 1
        for _ in range(b): self.elements[el_idx] = {'s': 3, 'e': 2, 'deg': -1, 'len': 2}; self.y_s_ids.append(el_idx); el_idx += 1
        for _ in range(c): self.elements[el_idx] = {'s': 1, 'e': 3, 'deg': -1, 'len': 2}; self.z_s_ids.append(el_idx); el_idx += 1

        for v in [1, 2, 3]: self.elements[el_idx] = {'s': v, 'e': v, 'deg': -2, 'len': 3}; self.t_ids.append(el_idx); el_idx += 1

    def generate_potential(self, potential_type='generic', seed=None):
        """
        ポテンシャル W を自動生成するメソッド。
        任意の (a,b,c) に対して、規則的で特別な形のポテンシャルを生成できるように一般化しました。
        """
        a, b, c = self.a, self.b, self.c
        C = {}
        # 各変数の共通して取れるインデックスの最大範囲
        N = min(a, b, c) 
        
        if potential_type == 'generic':
            if seed is not None:
                set_random_seed(seed)
            for i in range(a):
                for j in range(b):
                    for k in range(c):
                        C[(i, j, k)] = ZZ.random_element(-100, 101)
                        
        elif potential_type == 'diagonal':
            # 対角項のみを足し合わせる (例: x_0 y_0 z_0 + x_1 y_1 z_1 + ...)
            for i in range(N):
                C[(i, i, i)] = 1
                
        elif potential_type == 'dense_ones':
            # 考えうる全ての項の係数を 1 にする (ベタ塗り)
            for i in range(a):
                for j in range(b):
                    for k in range(c):
                        C[(i, j, k)] = 1
                        
        elif potential_type == 'alternating':
            # インデックスの和の偶奇で符号が変わる市松模様的なポテンシャル
            for i in range(a):
                for j in range(b):
                    for k in range(c):
                        C[(i, j, k)] = 1 if (i + j + k) % 2 == 0 else -1
                        
        elif potential_type == 'levi_civita':
            # 相異なるインデックスの組に対し、置換の符号(偶置換=1, 奇置換=-1)を係数とする
            # (旧 symmetric の完全な一般化)
            import itertools
            if N >= 3:
                # 0 から N-1 までの3つの相異なるインデックスを選ぶ
                for indices in itertools.permutations(range(N), 3):
                    i, j, k = indices
                    # 転倒数(inversions)を計算して偶奇を判定
                    inversions = 0
                    if i > j: inversions += 1
                    if i > k: inversions += 1
                    if j > k: inversions += 1
                    C[(i, j, k)] = 1 if inversions % 2 == 0 else -1
            else:
                # N < 3 の場合は反対称テンソルが作れないため diagonal にフォールバック
                for i in range(N):
                    C[(i, i, i)] = 1
                    
        elif potential_type == 'cyclic_shift':
            # 対角項を -1 とし、インデックスが巡回する項を +1 とする特異な形
            # (旧 singular の一般化)
            for i in range(N):
                C[(i, i, i)] = -1
            if N >= 2:
                for i in range(N):
                    # インデックスを1つずつずらす
                    j = (i + 1) % N
                    k = (i + 2) % N if N >= 3 else (i + 1) % N
                    C[(i, j, k)] = 1
                    
        return C

    def display_potential(self):
        """現在設定されているポテンシャル W を数式として出力する"""
        terms = []
        for i in range(self.a):
            for j in range(self.b):
                for k in range(self.c):
                    # 辞書に存在しないキーは係数0として扱う
                    coeff = self.W_coeffs.get((i, j, k), 0)
                    if coeff == 0:
                        continue
                    
                    term = f"x_{i} y_{j} z_{k}"
                    if coeff == 1:
                        terms.append(f"+ {term}")
                    elif coeff == -1:
                        terms.append(f"- {term}")
                    elif coeff > 0:
                        terms.append(f"+ {coeff}{term}")
                    else:
                        terms.append(f"- {abs(coeff)}{term}")
                        
        if not terms:
            print("W = 0")
        else:
            expr = " ".join(terms)
            if expr.startswith("+ "):
                expr = expr[2:] # 先頭の余分な + を削除
            print(f"W = {expr}")

    def get_basis(self, target_deg, target_len):
        """DFSによる基底探索 (キャッシュ付き)"""
        cache_key = (target_deg, target_len)
        if cache_key in self.basis_cache:
            return self.basis_cache[cache_key]
            
        basis = []
        def dfs(curr_node, curr_deg, curr_len, path):
            if curr_len == target_len:
                if curr_deg == target_deg: basis.append(tuple(path))
                return
            for el_id, el in self.elements.items():
                if el['s'] == curr_node and curr_deg + el['deg'] >= target_deg and curr_len + el['len'] <= target_len:
                    dfs(el['e'], curr_deg + el['deg'], curr_len + el['len'], path + [el_id])
                    
        for start in [1, 2, 3]: dfs(start, 0, 0, [])
        self.basis_cache[cache_key] = basis
        return basis

    def _get_derivations(self, p):
        """保持しているポテンシャル W から素数体 GF(p) 上での導分を計算"""
        F = GF(p)
        a, b, c = self.a, self.b, self.c
        
        # 整数の係数を GF(p) の要素に変換 (辞書にない項は0)
        C_F = {}
        for i in range(a):
            for j in range(b):
                for k in range(c):
                    val = self.W_coeffs.get((i, j, k), 0)
                    C_F[(i, j, k)] = F(val)

        derivations = {el_id: [] for el_id in self.elements}
        for i in range(a): derivations[self.x_s_ids[i]] = [(C_F[(i, j, k)], [self.y_ids[j], self.z_ids[k]]) for j in range(b) for k in range(c) if C_F[(i, j, k)] != 0]
        for j in range(b): derivations[self.y_s_ids[j]] = [(C_F[(i, j, k)], [self.z_ids[k], self.x_ids[i]]) for i in range(a) for k in range(c) if C_F[(i, j, k)] != 0]
        for k in range(c): derivations[self.z_s_ids[k]] = [(C_F[(i, j, k)], [self.x_ids[i], self.y_ids[j]]) for i in range(a) for j in range(b) if C_F[(i, j, k)] != 0]

        derivations[self.t_ids[0]] = [(F(1), [self.x_ids[i], self.x_s_ids[i]]) for i in range(a)] + [(F(-1), [self.z_s_ids[k], self.z_ids[k]]) for k in range(c)]
        derivations[self.t_ids[1]] = [(F(1), [self.y_ids[j], self.y_s_ids[j]]) for j in range(b)] + [(F(-1), [self.x_s_ids[i], self.x_ids[i]]) for i in range(a)]
        derivations[self.t_ids[2]] = [(F(1), [self.z_ids[k], self.z_s_ids[k]]) for k in range(c)] + [(F(-1), [self.y_s_ids[j], self.y_ids[j]]) for j in range(b)]

        return derivations

    def _diff_path(self, path, derivations):
        """パスの微分を計算"""
        res = []
        sign = 1
        for i, el_id in enumerate(path):
            for c_val, term_path in derivations[el_id]:
                res.append((c_val * sign, tuple(path[:i]) + tuple(term_path) + tuple(path[i+1:])))
            if self.elements[el_id]['deg'] % 2 != 0: sign *= -1
        return res

    def _calculate_over_prime(self, p, target_k, L):
        """コア計算エンジン: 疎行列(Sparse Matrix)を用いてランクを計算"""
        F = GF(p)
        V_prev = self.get_basis(-target_k + 1, L)
        V_curr = self.get_basis(-target_k, L)
        V_next = self.get_basis(-target_k - 1, L)
        
        if len(V_curr) == 0:
            return 0, 0, 0, len(V_prev), len(V_curr), len(V_next)

        V_prev_map = {path: i for i, path in enumerate(V_prev)}
        V_curr_map = {path: i for i, path in enumerate(V_curr)}

        derivations = self._get_derivations(p)

        # 行列 D_{-k} の計算
        curr_entries = {}
        for j, path in enumerate(V_curr):
            for c_val, res_path in self._diff_path(path, derivations):
                if res_path in V_prev_map:
                    r = V_prev_map[res_path]
                    curr_entries[(r, j)] = curr_entries.get((r, j), F(0)) + c_val
                    
        mat_curr = matrix(F, len(V_prev), len(V_curr), curr_entries, sparse=True)
        rank_curr = mat_curr.rank()
        dim_ker = len(V_curr) - rank_curr

        # 行列 D_{-(k+1)} の計算
        rank_next = 0
        if len(V_next) > 0:
            next_entries = {}
            for j, path in enumerate(V_next):
                for c_val, res_path in self._diff_path(path, derivations):
                    if res_path in V_curr_map:
                        r = V_curr_map[res_path]
                        next_entries[(r, j)] = next_entries.get((r, j), F(0)) + c_val
            mat_next = matrix(F, len(V_curr), len(V_next), next_entries, sparse=True)
            rank_next = mat_next.rank()

        dim_H = dim_ker - rank_next
        return dim_H, rank_curr, rank_next, len(V_prev), len(V_curr), len(V_next)

    def compute_single_cohomology(self, target_k, L, verbose=True):
        """
        指定した次数 k と長さ L のコホモロジー群の次元を計算する。
        """
            
        dim_1, r_c1, r_n1, v_prev, v_curr, v_next = self._calculate_over_prime(32003, target_k, L)
        dim_2, _, _, _, _, _ = self._calculate_over_prime(31991, target_k, L)
        final_dim = min(dim_1, dim_2)
        
        if verbose:
            print(f"--- Computing H^{{{target_k}}} at Length L={L} ({method} mode) ---")
            print(f"  V_{{{target_k+1}}} dim: {v_prev}")
            print(f"  V_{{{target_k}}} dim: {v_curr}")
            print(f"  V_{{{target_k-1}}} dim: {v_next}")
            print(f"  rank(D_curr) = {r_c1}, rank(D_next) = {r_n1}")
            print(f"  => dim H^{{{target_k}}}_L = {final_dim}\n")
            
        return final_dim

    def generate_collapse_map(self, max_k=4, max_L=8):
        """H^{-k}_L の一覧表（Collapse Map）を生成する"""
        print(f"=== Generating Collapse Map for Q({self.a},{self.b},{self.c}) [{self.potential_type.upper()}] ===")
        print(f"Target: k = 1 to {max_k}, Length L = 2 to {max_L}\n")
        
        results = {}
        for target_k in range(1, max_k + 1):
            for L in range(2, max_L + 1):
                print(f"Computing H^{{-{target_k}}}_L={L}... ", end="", flush=True)
                if L < target_k + 1:
                    results[(target_k, L)] = 0
                    print("Skipped (Empty space)")
                    continue
                    
                dim_1, _, _, _, v_curr, _ = self._calculate_over_prime(32003, target_k, L)
                if v_curr == 0 or dim_1 == 0:
                    results[(target_k, L)] = 0
                    print("0")
                else:
                    print(f"{dim_1} (Checking GF(31991))... ", end="", flush=True)
                    dim_2, _, _, _, _, _ = self._calculate_over_prime(31991, target_k, L)
                    final_dim = min(dim_1, dim_2)
                    results[(target_k, L)] = final_dim
                    print(f"Confirmed: {final_dim}")

        print(f"\n=== The Collapse Map of Q({self.a},{self.b},{self.c}) [{self.potential_type.upper()}] ===")
        header = f"| Cohomology \\ Length | " + " | ".join([f"L={L}" for L in range(2, max_L + 1)]) + " |"
        print(header)
        print("|---" + "|---" * max_L + "|")
        for target_k in range(1, max_k + 1):
            row = f"| **H^{{-{target_k}}}** | "
            for L in range(2, max_L + 1):
                val = results.get((target_k, L), 0)
                row += f"**{val}** | " if val > 0 else "0 | "
            print(row)
        print("\n")

    def compute_H0_dim(self, max_L):
        """
        L=0 から max_L までの H^0 の次元を厳密・超高速に計算する。
        パスの履歴は保持しつつ、最も重い検索用辞書は使い捨て（GCに回収させる）にして
        メモリ使用量を最小限に抑える差分ビルド実装。
        """
        if not hasattr(self, '_h0_cache_paths'):
            self._h0_cache_paths = {0: [(1,), (2,), (3,)]}
            a_list = []
            for i in range(self.a): a_list.append(('x', i, 1, 2))
            for j in range(self.b): a_list.append(('y', j, 2, 3))
            for k in range(self.c): a_list.append(('z', k, 3, 1))
            self._h0_a_list = a_list
            self._h0_cache_paths[1] = [((a[0], a[1]),) for a in a_list]
            
            self._h0_cache_dims = [3]
            if max_L >= 1: self._h0_cache_dims.append(len(self._h0_cache_paths[1]))
            
            base_rels = []
            for i in range(self.a):
                terms = [(self.W_coeffs[(i,j,k)], ('y', j), ('z', k)) for j in range(self.b) for k in range(self.c) if (i, j, k) in self.W_coeffs]
                if terms: base_rels.append((2, 1, terms))
            for j in range(self.b):
                terms = [(self.W_coeffs[(i,j,k)], ('z', k), ('x', i)) for i in range(self.a) for k in range(self.c) if (i, j, k) in self.W_coeffs]
                if terms: base_rels.append((3, 2, terms))
            for k in range(self.c):
                terms = [(self.W_coeffs[(i,j,k)], ('x', i), ('y', j)) for i in range(self.a) for j in range(self.b) if (i, j, k) in self.W_coeffs]
                if terms: base_rels.append((1, 3, terms))
            self._h0_base_rels = base_rels
            
            self._h0_max_calculated_L = 1 if max_L >= 1 else 0

        if max_L <= self._h0_max_calculated_L:
            return self._h0_cache_dims[:max_L + 1]

        for L in range(self._h0_max_calculated_L + 1, max_L + 1):
            new_paths = []
            for p in self._h0_cache_paths[L-1]:
                last_arr = p[-1]
                end_v = 2 if last_arr[0] == 'x' else 3 if last_arr[0] == 'y' else 1
                for a in self._h0_a_list:
                    if a[2] == end_v:
                        new_paths.append(p + ((a[0], a[1]),))
            
            self._h0_cache_paths[L] = new_paths
            col_size = len(new_paths)
            
            # 【重要】辞書をローカル変数にする。
            # ループを抜けるとPythonのガベージコレクタが自動的にメモリを解放する！
            current_path_idx = {p: i for i, p in enumerate(new_paths)}
            
            entries = {}
            row_idx = 0

            for L_p in range(L - 1):
                L_q = L - 2 - L_p
                for p in self._h0_cache_paths[L_p]:
                    end_p = p[0] if L_p == 0 else (2 if p[-1][0] == 'x' else 3 if p[-1][0] == 'y' else 1)
                    for start_r, end_r, terms in self._h0_base_rels:
                        if end_p == start_r:
                            for q in self._h0_cache_paths[L_q]:
                                start_q = q[0] if L_q == 0 else (1 if q[0][0] == 'x' else 2 if q[0][0] == 'y' else 3)
                                if end_r == start_q:
                                    has_terms = False
                                    for coeff, r1, r2 in terms:
                                        full_path = (p if L_p > 0 else ()) + (r1, r2) + (q if L_q > 0 else ())
                                        if full_path in current_path_idx:
                                            col = current_path_idx[full_path]
                                            entries[(row_idx, col)] = entries.get((row_idx, col), 0) + coeff
                                            has_terms = True
                                    if has_terms:
                                        row_idx += 1

            if row_idx == 0:
                self._h0_cache_dims.append(col_size)
            else:
                M = matrix(GF(31991), row_idx, col_size, entries, sparse=True)
                self._h0_cache_dims.append(col_size - M.rank())
                
            # メモリ解放を早めるため、明示的に削除してもOK
            del current_path_idx

        self._h0_max_calculated_L = max_L
        return self._h0_cache_dims[:max_L + 1]


    #うまく動かないので後で削除
    def compute_H0_dim_fast_pruning(self, max_L):
        """
        行列の次元爆発とSingularのバグを完全に回避する究極のアルゴリズム。
        小さな行列の掃き出し法から正確に禁止ワードを抽出し、
        パスを生成する端から枝刈りすることで、L=20などの深層へ一瞬で到達します。
        """
        F = GF(31991)
        
        # 1. 長さ2のパスのリスト
        paths_yz = [(j, k) for j in range(self.b) for k in range(self.c)]
        paths_zx = [(k, i) for k in range(self.c) for i in range(self.a)]
        paths_xy = [(i, j) for i in range(self.a) for j in range(self.b)]
        
        # 2. 掃き出し法(echelonize)で正確な「禁止ワード(Leading Terms)」を11個抽出
        M_x = matrix(F, self.a, len(paths_yz))
        for i in range(self.a):
            for col, (j, k) in enumerate(paths_yz):
                if (i, j, k) in self.W_coeffs: M_x[i, col] = F(self.W_coeffs[(i, j, k)])
        M_x.echelonize()
        forbidden_len2 = { ('y', paths_yz[c][0], 'z', paths_yz[c][1]) for c in M_x.pivots() }

        M_y = matrix(F, self.b, len(paths_zx))
        for j in range(self.b):
            for col, (k, i) in enumerate(paths_zx):
                if (i, j, k) in self.W_coeffs: M_y[j, col] = F(self.W_coeffs[(i, j, k)])
        M_y.echelonize()
        forbidden_len2.update({ ('z', paths_zx[c][0], 'x', paths_zx[c][1]) for c in M_y.pivots() })

        M_z = matrix(F, self.c, len(paths_xy))
        for k in range(self.c):
            for col, (i, j) in enumerate(paths_xy):
                if (i, j, k) in self.W_coeffs: M_z[k, col] = F(self.W_coeffs[(i, j, k)])
        M_z.echelonize()
        forbidden_len2.update({ ('x', paths_xy[c][0], 'y', paths_xy[c][1]) for c in M_z.pivots() })

        # 3. 繋がらない矢印のペア（81個）も禁止ワードに追加
        a_list = []
        for i in range(self.a): a_list.append(('x', i, 1, 2))
        for j in range(self.b): a_list.append(('y', j, 2, 3))
        for k in range(self.c): a_list.append(('z', k, 3, 1))
        
        for a1 in a_list:
            for a2 in a_list:
                # 【修正箇所】a1の終点(3) と a2の始点(2) を正しく比較！
                if a1[3] != a2[2]: 
                    forbidden_len2.add((a1[0], a1[1], a2[0], a2[1]))

        # 4. 枝刈りしながらパスを構築（次元爆発が起きない！）
        dims = [3]
        if max_L >= 1: dims.append(len(a_list))
        
        valid_paths = [ (a[0], a[1]) for a in a_list ]
        
        for L in range(2, max_L + 1):
            new_paths = []
            for p in valid_paths:
                last_kind = p[-2]
                end_v = 2 if last_kind == 'x' else 3 if last_kind == 'y' else 1
                
                for a in a_list:
                    if a[2] == end_v:
                        # 新しいパスの末尾2文字が禁止ワードでないかチェック
                        tail = (p[-2], p[-1], a[0], a[1])
                        if tail not in forbidden_len2:
                            new_paths.append(p + (a[0], a[1]))
            
            valid_paths = new_paths
            dims.append(len(valid_paths))
            
        return dims
        
        
    def verify_3cy_with_actual_h0(self, max_L=7):
        """
        オイラー標数(漸化式)と実際のH^0の次元を比較し、
        ズレを検知することで3-CY性を厳密に検証する。
        """
        import numpy as np
        
        A = np.array([[0, self.a, 0], [0, 0, self.b], [self.c, 0, 0]], dtype=object)
        B = np.array([[0, 0, self.c], [self.a, 0, 0], [0, self.b, 0]], dtype=object)
        I = np.eye(3, dtype=object)
        
        H = [I, A, np.dot(A, A) - B]
        
        print(f"=== Q({self.a},{self.b},{self.c}) [{self.potential_type.upper()}] 高速厳密検証 (L=0~{max_L}) ===")
        
        # ここで Singular を使って L=0~max_L までの次元を一気に計算！
        actual_dims = self.compute_H0_dim(max_L)
        
        is_3cy = True
        for L in range(max_L + 1):
            # 漸化式からの理論値
            if L >= 3:
                H_next = np.dot(A, H[L-1]) - np.dot(B, H[L-2]) + H[L-3]
                H.append(H_next)
            euler_char = np.sum(H[L])
            
            # 実際のヤコビ代数の次元（リストから取得するだけ）
            actual_dim = actual_dims[L]
            
            # 比較
            if euler_char == actual_dim:
                status = "✅ 一致"
            else:
                status = f"❌ ズレ発生！ (差: {actual_dim - euler_char})"
                is_3cy = False
                
            print(f"L={L} | 漸化式(χ): {euler_char:4} | 実際のH^0: {actual_dim:4} | {status}")
            
            if not is_3cy:
                print(f"\n--> 【結論】L={L} で実際の次元が漸化式を {actual_dim - euler_char} 上回りました。")
                print(f"    これは確実に負のコホモロジー（H^-1, H^-2 など）が存在する証拠であり、")
                print(f"    この代数は 3-CY ではありません！")
                break
                
        if is_3cy:
            print(f"\n--> 【結論】L={max_L} までズレは検出されませんでした。")
            print(f"    線形代数アプローチの計算範囲内では、3-CY である可能性が高いです。")

In [77]:
calc_1110.verify_3cy_with_actual_h0(max_L=7)

=== Q(3,4,4) [GENERIC] 高速厳密検証 (L=0~7) ===
L=0 | 漸化式(χ):    3 | 実際のH**Integer(0):    3 | ✅ 一致
L=1 | 漸化式(χ):   11 | 実際のH**Integer(0):   11 | ✅ 一致
L=2 | 漸化式(χ):   29 | 実際のH**Integer(0):   29 | ✅ 一致
L=3 | 漸化式(χ):   65 | 実際のH**Integer(0):   65 | ✅ 一致
L=4 | 漸化式(χ):  139 | 実際のH**Integer(0):  139 | ✅ 一致
L=5 | 漸化式(χ):  301 | 実際のH**Integer(0):  301 | ✅ 一致
L=6 | 漸化式(χ):  623 | 実際のH**Integer(0):  623 | ✅ 一致
L=7 | 漸化式(χ): 1280 | 実際のH**Integer(0): 1280 | ✅ 一致

--> 【結論】L=7 までズレは検出されませんでした。
    線形代数アプローチの計算範囲内では、3-CY である可能性が高いです。


In [ ]:
calc_1110.verify_3cy_with_actual_h0(max_L=8)

=== Q(3,4,4) [GENERIC] 高速厳密検証 (L=0~8) ===


In [2]:
import random

def search_non_vanishing_cohomology(a_range, b_range, c_range, max_k=4, max_L=8, potential_type='generic', base_seed=None):
    """
    コホモロジーが消えない箇所を探索し、発見した結果をリストとして返す。
    戻り値: [{'a': a, 'b': b, 'c': c, 'seed': seed, 'k': k, 'L': L, 'dim': dim}, ...]
    """
    print(f"=== 探索開始: k=1~{max_k}, L=2~{max_L}, potential_type='{potential_type}' ===")
    
    results = [] # 発見した結果を格納するリスト
    all_zero_results = []
    
    for a in a_range:
        for b in b_range:
            for c in c_range:

                if ( b < a ) or ( c < b ):
                    continue
                current_seed = base_seed if base_seed is not None else random.randint(0, 99999999)
                calc = GinzburgCohomologyCalculator(a, b, c, potential_type=potential_type, seed=current_seed)
                
                found_non_zero = False
                
                for target_k in range(1, max_k + 1):
                    if found_non_zero:
                        break 
                        
                    for L in range(2, max_L + 1):
                        if L < target_k + 1:
                            continue 
                        
                        dim_1, _, _, _, v_curr, _ = calc._calculate_over_prime(32003, target_k, L)
                        
                        if v_curr > 0 and dim_1 > 0:
                            dim_2, _, _, _, _, _ = calc._calculate_over_prime(31991, target_k, L)
                            final_dim = min(dim_1, dim_2)
                            
                            if final_dim > 0:
                                print(f"Q({a}, {b}, {c}) [Seed: {current_seed}] -> H^{{-{target_k}}}_{{{L}}} = {final_dim} (Non-zero!)")
                                
                                # 発見したデータを辞書としてリストに追加
                                results.append({
                                    'a': a, 'b': b, 'c': c,
                                    'seed': current_seed,
                                    'k': target_k,
                                    'L': L,
                                    'dim': final_dim
                                })
                                
                                found_non_zero = True
                                break
                                
                if not found_non_zero:
                    print(f"Q({a}, {b}, {c}) [Seed: {current_seed}] -> All H^{{-k}}_{{L}} = 0 (up to k={max_k}, L={max_L})")

                    # データを辞書としてリストに追加
                    all_zero_results.append({
                                    'a': a, 'b': b, 'c': c,
                                    'seed': current_seed,
                                    'k': target_k,
                                    'L': L,
                                    'dim': 0
                                })
                                
    print(f"=== 探索完了: {len(results)} 件の non-zero を発見 ===")
    return results, all_zero_results

In [3]:
def get_potential_expression(a, b, c, W_coeffs):
    """現在設定されているポテンシャル W を数式として"""
    terms = []
    for i in range(a):
        for j in range(b):
            for k in range(c):
                # 辞書に存在しないキーは係数0として扱う
                coeff = W_coeffs.get((i, j, k), 0)
                if coeff == 0:
                    continue
                
                term = f"x_{i} y_{j} z_{k}"
                if coeff == 1:
                    terms.append(f"+ {term}")
                elif coeff == -1:
                    terms.append(f"- {term}")
                elif coeff > 0:
                    terms.append(f"+ {coeff}{term}")
                else:
                    terms.append(f"- {abs(coeff)}{term}")
                    
    if not terms:
        expr = "0"
    else:
        expr = " ".join(terms)
        if expr.startswith("+ "):
            expr = expr[2:] # 先頭の余分な + を削除
    return expr
            

def get_potential_from_seed(a, b, c, seed, potential_type='generic'):
    """
    指定したパラメータとシード値から、計算に使用されたポテンシャルを再現して取得する。
    """
    calc = GinzburgCohomologyCalculator(a, b, c, potential_type=potential_type, seed=seed)
    
    # 内部でポテンシャルが格納されている変数名（Wなど）に合わせて調整してください
    W_coeffs = calc.W_coeffs  
    
    return get_potential_expression(a, b, c, W_coeffs)

In [ ]:
# 探索を実行し、結果のリストを受け取る
non_zero_results = search_non_vanishing_cohomology(
    a_range=range(1, 11), 
    b_range=range(1, 11), 
    c_range=range(1, 11), 
    max_k=2,       
    max_L=6
)
non_zero_results

=== 探索開始: k=1~2, L=2~6, potential_type='generic' ===
Q(1, 1, 1) [Seed: 78581238] -> H^{-2}_{4} = 3 (Non-zero!)
Q(1, 1, 2) [Seed: 65420826] -> H^{-1}_{2} = 1 (Non-zero!)
Q(1, 1, 3) [Seed: 29879462] -> H^{-1}_{2} = 2 (Non-zero!)
Q(1, 1, 4) [Seed: 9828183] -> H^{-1}_{2} = 3 (Non-zero!)
Q(1, 1, 5) [Seed: 81165130] -> H^{-1}_{2} = 4 (Non-zero!)
Q(1, 1, 6) [Seed: 4832258] -> H^{-1}_{2} = 5 (Non-zero!)
Q(1, 1, 7) [Seed: 21101590] -> H^{-1}_{2} = 6 (Non-zero!)
Q(1, 1, 8) [Seed: 25566633] -> H^{-1}_{2} = 7 (Non-zero!)
Q(1, 1, 9) [Seed: 11793276] -> H^{-1}_{2} = 8 (Non-zero!)
Q(1, 1, 10) [Seed: 89950271] -> H^{-1}_{2} = 9 (Non-zero!)
Q(1, 2, 2) [Seed: 3346157] -> H^{-1}_{3} = 3 (Non-zero!)
Q(1, 2, 3) [Seed: 23807591] -> H^{-1}_{2} = 1 (Non-zero!)
Q(1, 2, 4) [Seed: 88629945] -> H^{-1}_{2} = 2 (Non-zero!)
Q(1, 2, 5) [Seed: 99644584] -> H^{-1}_{2} = 3 (Non-zero!)
Q(1, 2, 6) [Seed: 47906713] -> H^{-1}_{2} = 4 (Non-zero!)
Q(1, 2, 7) [Seed: 34141174] -> H^{-1}_{2} = 5 (Non-zero!)
Q(1, 2, 8) [Seed: 237

IOStream.flush timed out


In [ ]:
# 探索を実行し、結果のリストを受け取る
non_zero_results, all_zero_results = search_non_vanishing_cohomology(
    a_range=range(1, 7), 
    b_range=range(1, 7), 
    c_range=range(1, 7), 
    max_k=2,       
    max_L=6
)

=== 探索開始: k=1~2, L=2~6, potential_type='generic' ===
Q(1, 1, 1) [Seed: 93426000] -> H^{-2}_{4} = 3 (Non-zero!)
Q(1, 1, 2) [Seed: 568179] -> H^{-1}_{2} = 1 (Non-zero!)
Q(1, 1, 3) [Seed: 87177796] -> H^{-1}_{2} = 2 (Non-zero!)
Q(1, 1, 4) [Seed: 53519913] -> H^{-1}_{2} = 3 (Non-zero!)
Q(1, 1, 5) [Seed: 20418020] -> H^{-1}_{2} = 4 (Non-zero!)
Q(1, 1, 6) [Seed: 56538711] -> H^{-1}_{2} = 5 (Non-zero!)
Q(1, 2, 2) [Seed: 37440993] -> H^{-1}_{3} = 3 (Non-zero!)
Q(1, 2, 3) [Seed: 4735725] -> H^{-1}_{2} = 1 (Non-zero!)
Q(1, 2, 4) [Seed: 59130383] -> H^{-1}_{2} = 2 (Non-zero!)
Q(1, 2, 5) [Seed: 32857711] -> H^{-1}_{2} = 3 (Non-zero!)
Q(1, 2, 6) [Seed: 37712366] -> H^{-1}_{2} = 4 (Non-zero!)
Q(1, 3, 3) [Seed: 88027496] -> H^{-1}_{3} = 8 (Non-zero!)
Q(1, 3, 4) [Seed: 3951386] -> H^{-1}_{2} = 1 (Non-zero!)
Q(1, 3, 5) [Seed: 70012143] -> H^{-1}_{2} = 2 (Non-zero!)
Q(1, 3, 6) [Seed: 9691279] -> H^{-1}_{2} = 3 (Non-zero!)
Q(1, 4, 4) [Seed: 92316509] -> H^{-1}_{3} = 15 (Non-zero!)
Q(1, 4, 5) [Seed: 92618

In [4]:
# 探索を実行し、結果のリストを受け取る
non_zero_results, all_zero_results = search_non_vanishing_cohomology(
    a_range=range(1, 6), 
    b_range=range(1, 6), 
    c_range=range(1, 6), 
    max_k=2,       
    max_L=6
)

=== 探索開始: k=1~2, L=2~6, potential_type='generic' ===
Q(1, 1, 1) [Seed: 24822820] -> H^{-2}_{4} = 3 (Non-zero!)
Q(1, 1, 2) [Seed: 95267150] -> H^{-1}_{2} = 1 (Non-zero!)
Q(1, 1, 3) [Seed: 44700641] -> H^{-1}_{2} = 2 (Non-zero!)
Q(1, 1, 4) [Seed: 26227850] -> H^{-1}_{2} = 3 (Non-zero!)
Q(1, 1, 5) [Seed: 21310386] -> H^{-1}_{2} = 4 (Non-zero!)
Q(1, 2, 2) [Seed: 96813029] -> H^{-1}_{3} = 3 (Non-zero!)
Q(1, 2, 3) [Seed: 27139184] -> H^{-1}_{2} = 1 (Non-zero!)
Q(1, 2, 4) [Seed: 95875922] -> H^{-1}_{2} = 2 (Non-zero!)
Q(1, 2, 5) [Seed: 22270671] -> H^{-1}_{2} = 3 (Non-zero!)
Q(1, 3, 3) [Seed: 31622854] -> H^{-1}_{3} = 8 (Non-zero!)
Q(1, 3, 4) [Seed: 66726707] -> H^{-1}_{2} = 1 (Non-zero!)
Q(1, 3, 5) [Seed: 59425741] -> H^{-1}_{2} = 2 (Non-zero!)
Q(1, 4, 4) [Seed: 36287032] -> H^{-1}_{3} = 15 (Non-zero!)
Q(1, 4, 5) [Seed: 60068633] -> H^{-1}_{2} = 1 (Non-zero!)
Q(1, 5, 5) [Seed: 36320844] -> H^{-1}_{3} = 24 (Non-zero!)
Q(2, 2, 2) [Seed: 95235406] -> H^{-1}_{3} = 3 (Non-zero!)
Q(2, 2, 3) [Seed:

In [5]:
non_zero_results

[{'a': 1, 'b': 1, 'c': 1, 'seed': 24822820, 'k': 2, 'L': 4, 'dim': 3},
 {'a': 1, 'b': 1, 'c': 2, 'seed': 95267150, 'k': 1, 'L': 2, 'dim': 1},
 {'a': 1, 'b': 1, 'c': 3, 'seed': 44700641, 'k': 1, 'L': 2, 'dim': 2},
 {'a': 1, 'b': 1, 'c': 4, 'seed': 26227850, 'k': 1, 'L': 2, 'dim': 3},
 {'a': 1, 'b': 1, 'c': 5, 'seed': 21310386, 'k': 1, 'L': 2, 'dim': 4},
 {'a': 1, 'b': 2, 'c': 2, 'seed': 96813029, 'k': 1, 'L': 3, 'dim': 3},
 {'a': 1, 'b': 2, 'c': 3, 'seed': 27139184, 'k': 1, 'L': 2, 'dim': 1},
 {'a': 1, 'b': 2, 'c': 4, 'seed': 95875922, 'k': 1, 'L': 2, 'dim': 2},
 {'a': 1, 'b': 2, 'c': 5, 'seed': 22270671, 'k': 1, 'L': 2, 'dim': 3},
 {'a': 1, 'b': 3, 'c': 3, 'seed': 31622854, 'k': 1, 'L': 3, 'dim': 8},
 {'a': 1, 'b': 3, 'c': 4, 'seed': 66726707, 'k': 1, 'L': 2, 'dim': 1},
 {'a': 1, 'b': 3, 'c': 5, 'seed': 59425741, 'k': 1, 'L': 2, 'dim': 2},
 {'a': 1, 'b': 4, 'c': 4, 'seed': 36287032, 'k': 1, 'L': 3, 'dim': 15},
 {'a': 1, 'b': 4, 'c': 5, 'seed': 60068633, 'k': 1, 'L': 2, 'dim': 1},
 {'a'

In [6]:
# 発見された結果について、一つずつポテンシャルを復元して確認する
print("\n--- 発見されたポテンシャルの詳細 ---")
for res in non_zero_results:
    a, b, c = res['a'], res['b'], res['c']
    seed = res['seed']
    
    # 用意した関数でポテンシャルを復元
    W = get_potential_from_seed(a, b, c, seed)
    
    print(f"\nQ({a}, {b}, {c}) [Seed: {seed}] のポテンシャル:")
    print(W)


--- 発見されたポテンシャルの詳細 ---

Q(1, 1, 1) [Seed: 24822820] のポテンシャル:
26x_0 y_0 z_0

Q(1, 1, 2) [Seed: 95267150] のポテンシャル:
- 98x_0 y_0 z_0 + 70x_0 y_0 z_1

Q(1, 1, 3) [Seed: 44700641] のポテンシャル:
- 5x_0 y_0 z_0 + 43x_0 y_0 z_1 + 34x_0 y_0 z_2

Q(1, 1, 4) [Seed: 26227850] のポテンシャル:
71x_0 y_0 z_0 + 2x_0 y_0 z_1 + 26x_0 y_0 z_2 + 53x_0 y_0 z_3

Q(1, 1, 5) [Seed: 21310386] のポテンシャル:
30x_0 y_0 z_0 + 54x_0 y_0 z_1 + 56x_0 y_0 z_2 - 67x_0 y_0 z_3 - 59x_0 y_0 z_4

Q(1, 2, 2) [Seed: 96813029] のポテンシャル:
63x_0 y_0 z_0 - 2x_0 y_0 z_1 - 32x_0 y_1 z_0 + 61x_0 y_1 z_1

Q(1, 2, 3) [Seed: 27139184] のポテンシャル:
- 27x_0 y_0 z_0 - 48x_0 y_0 z_1 - 75x_0 y_0 z_2 - 24x_0 y_1 z_0 + 93x_0 y_1 z_1 + 89x_0 y_1 z_2

Q(1, 2, 4) [Seed: 95875922] のポテンシャル:
15x_0 y_0 z_0 - 52x_0 y_0 z_1 + 61x_0 y_0 z_2 + 80x_0 y_0 z_3 - 38x_0 y_1 z_0 - 99x_0 y_1 z_1 - 15x_0 y_1 z_2

Q(1, 2, 5) [Seed: 22270671] のポテンシャル:
- 25x_0 y_0 z_0 + 68x_0 y_0 z_1 + 68x_0 y_0 z_2 + 25x_0 y_0 z_3 - 29x_0 y_0 z_4 + 22x_0 y_1 z_0 + 48x_0 y_1 z_1 + 11x_0 y_1 z_2 + 49x_0

In [7]:
all_zero_results

[{'a': 3, 'b': 3, 'c': 3, 'seed': 66556140, 'k': 2, 'L': 6, 'dim': 0},
 {'a': 3, 'b': 4, 'c': 4, 'seed': 61141987, 'k': 2, 'L': 6, 'dim': 0},
 {'a': 3, 'b': 4, 'c': 5, 'seed': 60851517, 'k': 2, 'L': 6, 'dim': 0},
 {'a': 3, 'b': 5, 'c': 5, 'seed': 80529777, 'k': 2, 'L': 6, 'dim': 0},
 {'a': 4, 'b': 4, 'c': 4, 'seed': 38653221, 'k': 2, 'L': 6, 'dim': 0},
 {'a': 4, 'b': 4, 'c': 5, 'seed': 30435254, 'k': 2, 'L': 6, 'dim': 0},
 {'a': 4, 'b': 5, 'c': 5, 'seed': 88245661, 'k': 2, 'L': 6, 'dim': 0},
 {'a': 5, 'b': 5, 'c': 5, 'seed': 1495673, 'k': 2, 'L': 6, 'dim': 0}]

In [8]:
# 発見された結果について、一つずつポテンシャルを復元して確認する
print("\n--- 発見されたポテンシャルの詳細 ---")
for res in all_zero_results:
    a, b, c = res['a'], res['b'], res['c']
    seed = res['seed']
    
    # 用意した関数でポテンシャルを復元
    W = get_potential_from_seed(a, b, c, seed)
    
    print(f"\nQ({a}, {b}, {c}) [Seed: {seed}] のポテンシャル:")
    print(W)


--- 発見されたポテンシャルの詳細 ---

Q(3, 3, 3) [Seed: 66556140] のポテンシャル:
49x_0 y_0 z_0 + 60x_0 y_0 z_1 + 2x_0 y_0 z_2 + 56x_0 y_1 z_0 + 65x_0 y_1 z_1 - 39x_0 y_1 z_2 - 53x_0 y_2 z_0 + 24x_0 y_2 z_1 + 47x_0 y_2 z_2 - 93x_1 y_0 z_0 + 99x_1 y_0 z_1 - 73x_1 y_0 z_2 + 33x_1 y_1 z_0 - 20x_1 y_1 z_1 + 5x_1 y_1 z_2 - 4x_1 y_2 z_0 + 13x_1 y_2 z_1 - 22x_1 y_2 z_2 - 90x_2 y_0 z_0 + 88x_2 y_0 z_1 + 88x_2 y_0 z_2 + 55x_2 y_1 z_0 - 17x_2 y_1 z_1 - 93x_2 y_1 z_2 + 61x_2 y_2 z_0 + 64x_2 y_2 z_1 + 20x_2 y_2 z_2

Q(3, 4, 4) [Seed: 61141987] のポテンシャル:
- 83x_0 y_0 z_0 - x_0 y_0 z_1 + 40x_0 y_0 z_2 + 81x_0 y_0 z_3 + 65x_0 y_1 z_0 - 11x_0 y_1 z_1 - 66x_0 y_1 z_2 - 75x_0 y_1 z_3 + 93x_0 y_2 z_0 + 34x_0 y_2 z_1 + 50x_0 y_2 z_2 + 27x_0 y_2 z_3 + 48x_0 y_3 z_0 + x_0 y_3 z_1 + 17x_0 y_3 z_2 + 35x_0 y_3 z_3 - 48x_1 y_0 z_0 - 74x_1 y_0 z_1 - 8x_1 y_0 z_2 + 45x_1 y_0 z_3 + 95x_1 y_1 z_0 + 48x_1 y_1 z_1 + 37x_1 y_1 z_2 - 6x_1 y_1 z_3 + 3x_1 y_2 z_0 - 72x_1 y_2 z_1 - 29x_1 y_2 z_2 - 34x_1 y_2 z_3 + 92x_1 y_3 z_0 + 27x_1 y_3 z_1 

---

In [4]:
calc_111 = GinzburgCohomologyCalculator(1, 1, 1)
calc_111.verify_3cy_with_actual_h0(max_L=6)

=== Q(1,1,1) [GENERIC] 高速厳密検証 (L=0~6) ===
L=0 | 漸化式(χ):    3 | 実際のH**Integer(0):    3 | ✅ 一致
L=1 | 漸化式(χ):    3 | 実際のH**Integer(0):    3 | ✅ 一致
L=2 | 漸化式(χ):    0 | 実際のH**Integer(0):    0 | ✅ 一致
L=3 | 漸化式(χ):    0 | 実際のH**Integer(0):    0 | ✅ 一致
L=4 | 漸化式(χ):    3 | 実際のH**Integer(0):    0 | ❌ ズレ発生！ (差: -3)

--> 【結論】L=4 で実際の次元が漸化式を -3 上回りました。
    これは確実に負のコホモロジー（H^-1, H^-2 など）が存在する証拠であり、
    この代数は 3-CY ではありません！


In [17]:
calc_111 = GinzburgCohomologyCalculator(3, 3, 4)
calc_111.verify_3cy_with_actual_h0(max_L=6)

=== Q(3,3,4) [GENERIC] 3-CY性の厳密検証 (L=0~6) ===
L=0 | 漸化式(χ):    3 | 実際のH**Integer(0):    3 | ✅ 一致
L=1 | 漸化式(χ):   10 | 実際のH**Integer(0):   10 | ✅ 一致
L=2 | 漸化式(χ):   23 | 実際のH**Integer(0):   23 | ✅ 一致
L=3 | 漸化式(χ):   43 | 実際のH**Integer(0):   43 | ✅ 一致
L=4 | 漸化式(χ):   73 | 実際のH**Integer(0):   73 | ✅ 一致
L=5 | 漸化式(χ):  125 | 実際のH**Integer(0):  126 | ❌ ズレ発生！ (差: 1)

--> 【結論】L=5 で実際の次元が漸化式を 1 上回りました。
    これは確実に負のコホモロジー（H^-1, H^-2 など）が存在する証拠であり、
    この代数は 3-CY ではありません！


In [19]:
calc_111 = GinzburgCohomologyCalculator(3, 4, 4)
calc_111.verify_3cy_with_actual_h0(max_L=7)

=== Q(3,4,4) [GENERIC] 3-CY性の厳密検証 (L=0~7) ===
L=0 | 漸化式(χ):    3 | 実際のH**Integer(0):    3 | ✅ 一致
L=1 | 漸化式(χ):   11 | 実際のH**Integer(0):   11 | ✅ 一致
L=2 | 漸化式(χ):   29 | 実際のH**Integer(0):   29 | ✅ 一致
L=3 | 漸化式(χ):   65 | 実際のH**Integer(0):   65 | ✅ 一致
L=4 | 漸化式(χ):  139 | 実際のH**Integer(0):  139 | ✅ 一致
L=5 | 漸化式(χ):  301 | 実際のH**Integer(0):  301 | ✅ 一致
L=6 | 漸化式(χ):  623 | 実際のH**Integer(0):  623 | ✅ 一致
L=7 | 漸化式(χ): 1280 | 実際のH**Integer(0): 1280 | ✅ 一致

--> 【結論】L=7 までズレは検出されませんでした。
    線形代数アプローチの計算範囲内では、3-CY である可能性が高いです。


In [21]:
calc_111 = GinzburgCohomologyCalculator(3, 3, 3)
calc_111.verify_3cy_with_actual_h0(max_L=7)

=== Q(3,3,3) [GENERIC] 3-CY性の厳密検証 (L=0~7) ===
L=0 | 漸化式(χ):    3 | 実際のH**Integer(0):    3 | ✅ 一致
L=1 | 漸化式(χ):    9 | 実際のH**Integer(0):    9 | ✅ 一致
L=2 | 漸化式(χ):   18 | 実際のH**Integer(0):   18 | ✅ 一致
L=3 | 漸化式(χ):   30 | 実際のH**Integer(0):   30 | ✅ 一致
L=4 | 漸化式(χ):   45 | 実際のH**Integer(0):   45 | ✅ 一致
L=5 | 漸化式(χ):   63 | 実際のH**Integer(0):   63 | ✅ 一致
L=6 | 漸化式(χ):   84 | 実際のH**Integer(0):   84 | ✅ 一致
L=7 | 漸化式(χ):  108 | 実際のH**Integer(0):  108 | ✅ 一致

--> 【結論】L=7 までズレは検出されませんでした。
    線形代数アプローチの計算範囲内では、3-CY である可能性が高いです。


# 漸化式

$Q$の隣接行列 $A$, potential $W$の微分から作った関係式の行列 $B$, 単位行列 $I$を用意する。（実際の定義は以下のcell参照）

H_Lを$3\times 3$行列で$(i,j)$成分がヤコビ代数 $J(Q,W)$内の長さ $L$の頂点$i$から始まって頂点$j$で終わるpathの成す空間の次元であるものとする。

Geminiによれば、ヤコビ代数 $J(Q,W)$が3-CYならば、ヤコビ代数の射影分解を追うことで
$$H_L = AH_{L-1}-BH_{L-2}+IH_{L-3}$$
が示されるらしい。※検証中

次のメソッドは粗い判定方法で、上の前回式を基に$H_L$を計算して負の成分が現れたら、次元が負になることはあり得ないので3-CYでないと判定しているものとなる。

In [10]:
import numpy as np

def check_hilbert_series(a, b, c, max_L=15):
    """
    コホモロジーが0次に集中している（3-Calabi-Yauである）と仮定した場合の、
    各長さ L のパスの理論上の次元を計算し、仮定が破綻しないかチェックする。
    """
    # 矢印の隣接行列 (0->1: a本, 1->2: b本, 2->0: c本)
    A = np.array([
        [0, a, 0],
        [0, 0, b],
        [c, 0, 0]
    ], dtype=object)
    
    # 関係式の行列 (微分によって逆向きのパスの数になる)
    B = np.array([
        [0, 0, c],
        [a, 0, 0],
        [0, b, 0]
    ], dtype=object)
    
    # 長さ3のポテンシャルからくる単位行列
    I = np.eye(3, dtype=object)
    
    # 初期条件
    H = [
        I,                           # L=0: 頂点のみ
        A,                           # L=1: 矢印のみ
        np.dot(A, A) - B             # L=2: 長さ2のパス - 関係式
    ]
    
    print(f"=== Q({a}, {b}, {c}) のヤコビ代数 理論次元の検証 ===")
    print("行: 始点 (0, 1, 2) / 列: 終点 (0, 1, 2)\n")
    
    is_valid = True
    
    for L in range(max_L + 1):
        if L >= 3:
            # オイラー・ポアンカレの原理に基づく漸化式
            H_next = np.dot(A, H[L-1]) - np.dot(B, H[L-2]) + H[L-3]
            H.append(H_next)
            
        dim_matrix = H[L]
        total_dim = np.sum(dim_matrix)
        
        # どこか一つでもマイナスの次元が発生したら、仮定（負のコホモロジーが0）は破綻
        has_negative = np.any(dim_matrix < 0)
        if has_negative:
            status = "❌ 負の次元発生！仮定が破綻（負のコホモロジーが存在します）"
            is_valid = False
        else:
            status = "✅"
            
        print(f"--- L = {L} (合計次元: {total_dim}) {status} ---")
        print(dim_matrix)
        
        if has_negative:
            break

    if is_valid:
        print(f"\n【結論】 L={max_L} まで検証しましたが、次元はすべて非負です。")
        print("これは「コホモロジーが0次にのみ集中している（ホモロジー的に滑らかである）」ことの極めて強力な証拠（Golod-Shafarevich型の等式成立）です。")



In [11]:
# 実行
check_hilbert_series(2, 2, 3, max_L=30)

=== Q(2, 2, 3) のヤコビ代数 理論次元の検証 ===
行: 始点 (0, 1, 2) / 列: 終点 (0, 1, 2)

--- L = 0 (合計次元: 3) ✅ ---
[[1 0 0]
 [0 1 0]
 [0 0 1]]
--- L = 1 (合計次元: 7) ✅ ---
[[0 2 0]
 [0 0 2]
 [3 0 0]]
--- L = 2 (合計次元: 9) ✅ ---
[[0 0 1]
 [4 0 0]
 [0 4 0]]
--- L = 3 (合計次元: 5) ✅ ---
[[0 0 0]
 [0 5 0]
 [0 0 0]]
--- L = 4 (合計次元: -5) ❌ 負の次元発生！仮定が破綻（負のコホモロジーが存在します） ---
[[0 0 0]
 [0 0 0]
 [-5 0 0]]


In [12]:
# 実行
check_hilbert_series(3, 4, 4, max_L=30)

=== Q(3, 4, 4) のヤコビ代数 理論次元の検証 ===
行: 始点 (0, 1, 2) / 列: 終点 (0, 1, 2)

--- L = 0 (合計次元: 3) ✅ ---
[[1 0 0]
 [0 1 0]
 [0 0 1]]
--- L = 1 (合計次元: 11) ✅ ---
[[0 3 0]
 [0 0 4]
 [4 0 0]]
--- L = 2 (合計次元: 29) ✅ ---
[[0 0 8]
 [13 0 0]
 [0 8 0]]
--- L = 3 (合計次元: 65) ✅ ---
[[24 0 0]
 [0 24 0]
 [0 0 17]]
--- L = 4 (合計次元: 139) ✅ ---
[[0 43 0]
 [0 0 48]
 [48 0 0]]
--- L = 5 (合計次元: 301) ✅ ---
[[0 0 84]
 [133 0 0]
 [0 84 0]]
--- L = 6 (合計次元: 623) ✅ ---
[[231 0 0]
 [0 231 0]
 [0 0 161]]
--- L = 7 (合計次元: 1280) ✅ ---
[[0 400 0]
 [0 0 440]
 [440 0 0]]
--- L = 8 (合計次元: 2720) ✅ ---
[[0 0 760]
 [1200 0 0]
 [0 760 0]]
--- L = 9 (合計次元: 5583) ✅ ---
[[2071 0 0]
 [0 2071 0]
 [0 0 1441]]
--- L = 10 (合計次元: 11421) ✅ ---
[[0 3573 0]
 [0 0 3924]
 [3924 0 0]]
--- L = 11 (合計次元: 24219) ✅ ---
[[0 0 6768]
 [10683 0 0]
 [0 6768 0]]
--- L = 12 (合計次元: 49665) ✅ ---
[[18424 0 0]
 [0 18424 0]
 [0 0 12817]]
--- L = 13 (合計次元: 101549) ✅ ---
[[0 31773 0]
 [0 0 34888]
 [34888 0 0]]
--- L = 14 (合計次元: 215291) ✅ ---
[[0 0 60164]
 [94963 0

In [13]:
# 実行
check_hilbert_series(4, 4, 4, max_L=30)

=== Q(4, 4, 4) のヤコビ代数 理論次元の検証 ===
行: 始点 (0, 1, 2) / 列: 終点 (0, 1, 2)

--- L = 0 (合計次元: 3) ✅ ---
[[1 0 0]
 [0 1 0]
 [0 0 1]]
--- L = 1 (合計次元: 12) ✅ ---
[[0 4 0]
 [0 0 4]
 [4 0 0]]
--- L = 2 (合計次元: 36) ✅ ---
[[0 0 12]
 [12 0 0]
 [0 12 0]]
--- L = 3 (合計次元: 99) ✅ ---
[[33 0 0]
 [0 33 0]
 [0 0 33]]
--- L = 4 (合計次元: 264) ✅ ---
[[0 88 0]
 [0 0 88]
 [88 0 0]]
--- L = 5 (合計次元: 696) ✅ ---
[[0 0 232]
 [232 0 0]
 [0 232 0]]
--- L = 6 (合計次元: 1827) ✅ ---
[[609 0 0]
 [0 609 0]
 [0 0 609]]
--- L = 7 (合計次元: 4788) ✅ ---
[[0 1596 0]
 [0 0 1596]
 [1596 0 0]]
--- L = 8 (合計次元: 12540) ✅ ---
[[0 0 4180]
 [4180 0 0]
 [0 4180 0]]
--- L = 9 (合計次元: 32835) ✅ ---
[[10945 0 0]
 [0 10945 0]
 [0 0 10945]]
--- L = 10 (合計次元: 85968) ✅ ---
[[0 28656 0]
 [0 0 28656]
 [28656 0 0]]
--- L = 11 (合計次元: 225072) ✅ ---
[[0 0 75024]
 [75024 0 0]
 [0 75024 0]]
--- L = 12 (合計次元: 589251) ✅ ---
[[196417 0 0]
 [0 196417 0]
 [0 0 196417]]
--- L = 13 (合計次元: 1542684) ✅ ---
[[0 514228 0]
 [0 0 514228]
 [514228 0 0]]
--- L = 14 (合計次元: 4038804

In [14]:
# 実行
check_hilbert_series(3, 3, 5, max_L=30)

=== Q(3, 3, 5) のヤコビ代数 理論次元の検証 ===
行: 始点 (0, 1, 2) / 列: 終点 (0, 1, 2)

--- L = 0 (合計次元: 3) ✅ ---
[[1 0 0]
 [0 1 0]
 [0 0 1]]
--- L = 1 (合計次元: 11) ✅ ---
[[0 3 0]
 [0 0 3]
 [5 0 0]]
--- L = 2 (合計次元: 28) ✅ ---
[[0 0 4]
 [12 0 0]
 [0 12 0]]
--- L = 3 (合計次元: 52) ✅ ---
[[12 0 0]
 [0 28 0]
 [0 0 12]]
--- L = 4 (合計次元: 83) ✅ ---
[[0 27 0]
 [0 0 27]
 [29 0 0]]
--- L = 5 (合計次元: 151) ✅ ---
[[0 0 25]
 [63 0 0]
 [0 63 0]]
--- L = 6 (合計次元: 248) ✅ ---
[[56 0 0]
 [0 136 0]
 [0 0 56]]
--- L = 7 (合計次元: 360) ✅ ---
[[0 120 0]
 [0 0 120]
 [120 0 0]]
--- L = 8 (合計次元: 615) ✅ ---
[[0 0 105]
 [255 0 0]
 [0 255 0]]
--- L = 9 (合計次元: 983) ✅ ---
[[221 0 0]
 [0 541 0]
 [0 0 221]]
--- L = 10 (合計次元: 1396) ✅ ---
[[0 468 0]
 [0 0 468]
 [460 0 0]]
--- L = 11 (合計次元: 2348) ✅ ---
[[0 0 404]
 [972 0 0]
 [0 972 0]]
--- L = 12 (合計次元: 3727) ✅ ---
[[837 0 0]
 [0 2053 0]
 [0 0 837]]
--- L = 13 (合計次元: 5263) ✅ ---
[[0 1767 0]
 [0 0 1767]
 [1729 0 0]]
--- L = 14 (合計次元: 8816) ✅ ---
[[0 0 1520]
 [3648 0 0]
 [0 3648 0]]
--- L = 15 (合計次元:

In [2]:
calc = GinzburgCohomologyCalculator(3, 4, 4)
calc.display_potential()
calc.generate_collapse_map(max_k=2, max_L=7)

W = - 61x_0 y_0 z_0 - 13x_0 y_0 z_1 + 93x_0 y_0 z_2 - 74x_0 y_0 z_3 + 11x_0 y_1 z_0 + 25x_0 y_1 z_1 + 67x_0 y_1 z_2 + 25x_0 y_1 z_3 + 99x_0 y_2 z_0 - 26x_0 y_2 z_1 - 92x_0 y_2 z_2 + 100x_0 y_2 z_3 + 86x_0 y_3 z_0 - 15x_0 y_3 z_1 - 77x_0 y_3 z_2 - 76x_0 y_3 z_3 + 92x_1 y_0 z_0 + 60x_1 y_0 z_2 + 99x_1 y_0 z_3 + 20x_1 y_1 z_1 - 81x_1 y_1 z_2 + 26x_1 y_1 z_3 + 34x_1 y_2 z_0 - 51x_1 y_2 z_1 + 25x_1 y_2 z_2 + 85x_1 y_2 z_3 - 43x_1 y_3 z_0 + 13x_1 y_3 z_1 + 32x_1 y_3 z_2 + 31x_1 y_3 z_3 - 51x_2 y_0 z_0 + 13x_2 y_0 z_1 + 91x_2 y_0 z_2 + 43x_2 y_0 z_3 - 87x_2 y_1 z_0 + 44x_2 y_1 z_1 + 63x_2 y_1 z_2 - 88x_2 y_1 z_3 + x_2 y_2 z_0 + 91x_2 y_2 z_1 + 16x_2 y_2 z_2 + 49x_2 y_2 z_3 + 28x_2 y_3 z_0 - 93x_2 y_3 z_1 + 66x_2 y_3 z_2 + 73x_2 y_3 z_3
=== Generating Collapse Map for Q(3,4,4) [GENERIC] ===
Target: k = 1 to 2, Length L = 2 to 7

Computing H^{-1}_L=2... 0
Computing H^{-1}_L=3... 0
Computing H^{-1}_L=4... 0
Computing H^{-1}_L=5... 0
Computing H^{-1}_L=6... 0
Computing H^{-1}_L=7... 0
Computing H

In [10]:
calc.compute_single_cohomology(3,8)

--- Computing H^{-3} at Length L=8 for Q(3,4,4) [GENERIC] ---
Dimensions: V_{-2}=122056, V_{-3}=31251, V_{-4}=2572
rank(D_{-3}) = 28712
rank(D_{-4}) = 2539
--> dim H^{-3}_L = 0  (Mathematically Guaranteed over Q!)



0

In [11]:
calc = GinzburgCohomologyCalculator(3, 3, 6)
calc.display_potential()
calc.generate_collapse_map(max_k=3, max_L=7)

W = 82x_0 y_0 z_0 - 32x_0 y_0 z_1 + 69x_0 y_0 z_2 - 9x_0 y_0 z_3 + 71x_0 y_0 z_4 + 8x_0 y_0 z_5 + 83x_0 y_1 z_0 + 67x_0 y_1 z_1 + 81x_0 y_1 z_2 - 17x_0 y_1 z_3 + 16x_0 y_1 z_4 - 22x_0 y_1 z_5 - 88x_0 y_2 z_0 + 55x_0 y_2 z_1 + 27x_0 y_2 z_2 + 58x_0 y_2 z_3 - 77x_0 y_2 z_4 - 98x_0 y_2 z_5 - 71x_1 y_0 z_0 + 32x_1 y_0 z_1 + 30x_1 y_0 z_2 - 97x_1 y_0 z_3 - 41x_1 y_0 z_4 + 93x_1 y_0 z_5 + 56x_1 y_1 z_0 + 38x_1 y_1 z_1 + 10x_1 y_1 z_2 + 10x_1 y_1 z_3 - 57x_1 y_1 z_4 - 84x_1 y_1 z_5 - 34x_1 y_2 z_0 - 92x_1 y_2 z_1 + 24x_1 y_2 z_2 + 73x_1 y_2 z_3 - 61x_1 y_2 z_4 - 82x_1 y_2 z_5 - 84x_2 y_0 z_0 - 49x_2 y_0 z_1 + 91x_2 y_0 z_2 - 5x_2 y_0 z_3 + 85x_2 y_0 z_4 + 89x_2 y_0 z_5 + 65x_2 y_1 z_0 - 66x_2 y_1 z_1 - 21x_2 y_1 z_2 - 47x_2 y_1 z_3 - 62x_2 y_1 z_4 - 29x_2 y_1 z_5 - 87x_2 y_2 z_0 - 84x_2 y_2 z_1 - 84x_2 y_2 z_2 - 92x_2 y_2 z_3 - 23x_2 y_2 z_4 + 29x_2 y_2 z_5
=== Generating Collapse Map for Q(3,3,6) [GENERIC] ===
Target: k = 1 to 3, Length L = 2 to 7

Computing H^{-1}_L=2... 0
Computing H^{-1}_

In [14]:
calc = GinzburgCohomologyCalculator(3, 4, 4)
calc.display_potential()
calc.generate_collapse_map(max_k=3, max_L=7)

W = 38x_0 y_0 z_0 + 96x_0 y_0 z_1 - 32x_0 y_0 z_2 - 14x_0 y_0 z_3 - 48x_0 y_1 z_0 + 78x_0 y_1 z_1 - 100x_0 y_1 z_2 - 2x_0 y_1 z_3 - 75x_0 y_2 z_0 + 35x_0 y_2 z_1 + 94x_0 y_2 z_2 - 23x_0 y_2 z_3 - 91x_0 y_3 z_0 + 39x_0 y_3 z_1 - 56x_0 y_3 z_2 - 53x_0 y_3 z_3 + 77x_1 y_0 z_0 + 19x_1 y_0 z_1 + 39x_1 y_0 z_2 + 60x_1 y_0 z_3 + 91x_1 y_1 z_0 + 14x_1 y_1 z_1 + 89x_1 y_1 z_2 + 97x_1 y_1 z_3 - 56x_1 y_2 z_0 - 55x_1 y_2 z_1 + 46x_1 y_2 z_2 + 20x_1 y_2 z_3 + 93x_1 y_3 z_0 + 44x_1 y_3 z_1 + 91x_1 y_3 z_2 + 3x_1 y_3 z_3 - 82x_2 y_0 z_0 - 92x_2 y_0 z_1 - 6x_2 y_0 z_2 - 84x_2 y_0 z_3 + 16x_2 y_1 z_0 - 57x_2 y_1 z_1 - 54x_2 y_1 z_2 - 26x_2 y_1 z_3 + 46x_2 y_2 z_0 + 89x_2 y_2 z_1 - 62x_2 y_2 z_2 + 86x_2 y_2 z_3 - 90x_2 y_3 z_0 - 56x_2 y_3 z_1 - 24x_2 y_3 z_2 - 23x_2 y_3 z_3
=== Generating Collapse Map for Q(3,4,4) [GENERIC] ===
Target: k = 1 to 3, Length L = 2 to 7

Computing H^{-1}_L=2... 0
Computing H^{-1}_L=3... 0
Computing H^{-1}_L=4... 0
Computing H^{-1}_L=5... 0
Computing H^{-1}_L=6... 0
Computin

In [5]:
calc = GinzburgCohomologyCalculator(1, 2, 2)
calc.display_potential()
calc.generate_collapse_map(max_k=3, max_L=7)

W = 15x_0 y_0 z_0 - 21x_0 y_0 z_1 + 90x_0 y_1 z_0 + 70x_0 y_1 z_1
=== Generating Collapse Map for Q(1,2,2) [GENERIC] ===
Target: k = 1 to 3, Length L = 2 to 7

Computing H^{-1}_L=2... 0
Computing H^{-1}_L=3... 3 (Checking GF(31991))... Confirmed: 3
Computing H^{-1}_L=4... 8 (Checking GF(31991))... Confirmed: 8
Computing H^{-1}_L=5... 5 (Checking GF(31991))... Confirmed: 5
Computing H^{-1}_L=6... 0
Computing H^{-1}_L=7... 0
Computing H^{-2}_L=2... Skipped (Empty space)
Computing H^{-2}_L=3... 0
Computing H^{-2}_L=4... 1 (Checking GF(31991))... Confirmed: 1
Computing H^{-2}_L=5... 4 (Checking GF(31991))... Confirmed: 4
Computing H^{-2}_L=6... 15 (Checking GF(31991))... Confirmed: 15
Computing H^{-2}_L=7... 24 (Checking GF(31991))... Confirmed: 24
Computing H^{-3}_L=2... Skipped (Empty space)
Computing H^{-3}_L=3... Skipped (Empty space)
Computing H^{-3}_L=4... 0
Computing H^{-3}_L=5... 0
Computing H^{-3}_L=6... 0
Computing H^{-3}_L=7... 0

=== The Collapse Map of Q(1,2,2) [GENERIC] ===
|

In [ ]:
import time
import csv
import numpy as np
import random  # 乱数シード生成用に追加

def run_3cy_screening(max_abc=7, max_L=7, output_file="screening_results_with_seed.csv"):
    """
    1 <= a <= b <= c <= max_abc の範囲で、
    L=0 から max_L までの 3-CY 性を全自動検証し、
    再現用のシード値と共に保存するスクリプト。
    """
    print(f"=== 3-CY Screening Started ===")
    print(f"Range: 1 <= a <= b <= c <= {max_abc}")
    print(f"Max Length: L={max_L}")
    print(f"Output: {output_file}\n")
    
    cases = []
    for a in range(1, max_abc + 1):
        for b in range(a, max_abc + 1):
            for c in range(b, max_abc + 1):
                cases.append((a, b, c))
                
    total_cases = len(cases)
    print(f"Total cases to test: {total_cases}")
    print("-" * 50)
    
    start_time = time.time()
    
    with open(output_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        # 【変更点】ヘッダーに Seed を追加
        writer.writerow(["a", "b", "c", "Status", "Broken_at_L", "Time(sec)", "Seed"])
        
        for idx, (a, b, c) in enumerate(cases):
            case_start = time.time()
            print(f"[{idx+1:02d}/{total_cases}] Testing Q({a},{b},{c})... ", end="", flush=True)
            
            # 【変更点】このケース用の固有のシードを生成
            current_seed = random.randint(1, 2**31 - 1)
            
            # 【変更点】生成したシードを計算機に渡す
            calc = GinzburgCohomologyCalculator(a, b, c, potential_type='generic', seed=current_seed)
            
            A = np.array([[0, a, 0], [0, 0, b], [c, 0, 0]], dtype=object)
            B = np.array([[0, 0, c], [a, 0, 0], [0, b, 0]], dtype=object)
            I = np.eye(3, dtype=object)
            H = [I, A, np.dot(A, A) - B]
            
            actual_dims = calc.compute_H0_dim(max_L)
            
            is_3cy = True
            broken_L = -1
            
            for L in range(max_L + 1):
                if L >= 3:
                    H_next = np.dot(A, H[L-1]) - np.dot(B, H[L-2]) + H[L-3]
                    H.append(H_next)
                euler_char = np.sum(H[L])
                actual_dim = actual_dims[L]
                
                if euler_char != actual_dim:
                    is_3cy = False
                    broken_L = L
                    break
                    
            case_time = time.time() - case_start
            
            if is_3cy:
                status = "✅ Candidate"
                broken_L_str = "None"
                print(f"✅ OK (Time: {case_time:.2f}s, Seed: {current_seed})")
            else:
                status = "❌ Broken"
                broken_L_str = str(broken_L)
                print(f"❌ Broken at L={broken_L} (Time: {case_time:.2f}s, Seed: {current_seed})")
                
            # 【変更点】CSVの行末にシード値を記録
            writer.writerow([a, b, c, status, broken_L_str, round(case_time, 2), current_seed])
            f.flush()
            
    total_time = time.time() - start_time
    print("-" * 50)
    print(f"=== Screening Completed ===")
    print(f"Total Time: {total_time:.2f}s")
    print(f"Results saved to {output_file}")

# ==========================================
# 実行コマンド
# ==========================================
run_3cy_screening(max_abc=7, max_L=7)

=== 3-CY Screening Started ===
Range: 1 <= a <= b <= c <= 7
Max Length: L=7
Output: screening_results_with_seed.csv

Total cases to test: 84
--------------------------------------------------
[01/84] Testing Q(1,1,1)... ❌ Broken at L=4 (Time: 0.10s, Seed: 723614350)
[02/84] Testing Q(1,1,2)... ❌ Broken at L=2 (Time: 0.01s, Seed: 1823603539)
[03/84] Testing Q(1,1,3)... ❌ Broken at L=2 (Time: 0.01s, Seed: 225445022)
[04/84] Testing Q(1,1,4)... ❌ Broken at L=2 (Time: 0.02s, Seed: 1772592742)
[05/84] Testing Q(1,1,5)... ❌ Broken at L=2 (Time: 0.02s, Seed: 1774216519)
[06/84] Testing Q(1,1,6)... ❌ Broken at L=2 (Time: 0.03s, Seed: 229337256)
[07/84] Testing Q(1,1,7)... ❌ Broken at L=2 (Time: 0.04s, Seed: 704799289)
[08/84] Testing Q(1,2,2)... ❌ Broken at L=3 (Time: 0.01s, Seed: 226027059)
[09/84] Testing Q(1,2,3)... ❌ Broken at L=2 (Time: 0.01s, Seed: 637040442)
[10/84] Testing Q(1,2,4)... ❌ Broken at L=2 (Time: 0.03s, Seed: 1664114289)
[11/84] Testing Q(1,2,5)... ❌ Broken at L=2 (Time: 0.0

In [ ]:
1+1